# Lesson 9: Evaluating Models

*ML Academy — companion notebook*

Confusion matrices, precision, recall, F1, ROC curves and regression metrics.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-09.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 12. In Python

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

X, y = load_breast_cancer(return_X_y=True)
y = 1 - y        # make "malignant" the positive class (1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]          # P(malignant)

for threshold in [0.5, 0.2]:
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    print(f"threshold {threshold}: TP={tp} FN={fn} FP={fp} TN={tn}  "
          f"precision={tp/(tp+fp):.3f} recall={tp/(tp+fn):.3f}")

print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
print(classification_report(y_test, (proba >= 0.5).astype(int), target_names=["benign", "malignant"]))

threshold 0.5: TP=51 FN=2 FP=0 TN=90  precision=1.000 recall=0.962
threshold 0.2: TP=52 FN=1 FP=9 TN=81  precision=0.852 recall=0.981
ROC-AUC: 0.996
              precision    recall  f1-score   support

      benign       0.98      1.00      0.99        90
   malignant       1.00      0.96      0.98        53

    accuracy                           0.99       143
   macro avg       0.99      0.98      0.98       143
weighted avg       0.99      0.99      0.99       143



## 15. Going deeper: comparing models honestly

In [2]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xtr, ytr)
p_lr, p_tr = logreg.predict_proba(Xte)[:, 1], tree.predict_proba(Xte)[:, 1]

rng = np.random.default_rng(0)
boot = []
for _ in range(2000):
    i = rng.integers(0, len(yte), len(yte))            # resample test cases with replacement
    if len(np.unique(yte[i])) == 2:
        boot.append(roc_auc_score(yte[i], p_lr[i]) - roc_auc_score(yte[i], p_tr[i]))
print("AUC: logistic", round(roc_auc_score(yte, p_lr), 3), " tree", round(roc_auc_score(yte, p_tr), 3))
print("95% bootstrap interval for the difference:", np.round(np.percentile(boot, [2.5, 97.5]), 3))

AUC: logistic 0.996  tree 0.904
95% bootstrap interval for the difference: [0.046 0.15 ]


In [3]:
from scipy import stats

# McNemar's test: do the two models make different numbers of mistakes on the SAME cases?
right_lr, right_tr = logreg.predict(Xte) == yte, tree.predict(Xte) == yte
b = np.sum(right_lr & ~right_tr)          # logistic right, tree wrong
c = np.sum(~right_lr & right_tr)          # tree right, logistic wrong
print("accuracy: logistic", round(right_lr.mean(), 3), " tree", round(right_tr.mean(), 3))
print("cases only logistic got right:", b, "  only the tree got right:", c)
print("McNemar exact p-value:", round(stats.binomtest(b, b + c, 0.5).pvalue, 4))

accuracy: logistic 0.959  tree 0.901
cases only logistic got right: 11   only the tree got right: 1
McNemar exact p-value: 0.0063


In [4]:
# Choosing the threshold from costs: a missed cancer (false negative) costs 20, a false alarm costs 1
y_bad = 1 - yte                            # in this dataset 0 = malignant, so flip: 1 = malignant
p_bad = 1 - p_lr
for t in [0.5, 0.2, 0.1, 0.05]:
    flag = p_bad >= t
    fn, fp = np.sum(~flag & (y_bad == 1)), np.sum(flag & (y_bad == 0))
    print(f"threshold {t:4.2f}: missed {fn:2d}, false alarms {fp:2d}, total cost {20 * fn + fp}")
print("theory: flag when p is above cost_FP / (cost_FP + cost_FN), which is", round(1 / 21, 3))

threshold 0.50: missed  3, false alarms  4, total cost 64
threshold 0.20: missed  2, false alarms  7, total cost 47
threshold 0.10: missed  1, false alarms 11, total cost 31
threshold 0.05: missed  0, false alarms 18, total cost 18
theory: flag when p is above cost_FP / (cost_FP + cost_FN), which is 0.048


In [5]:
from sklearn.metrics import average_precision_score

# Same model quality, rarer positives: scores of positives ~ N(1.5, 1), negatives ~ N(0, 1)
rng = np.random.default_rng(1)
for n_pos in [10000, 1000, 100]:
    s = np.r_[rng.normal(1.5, 1, n_pos), rng.normal(0, 1, 10000)]
    t = np.r_[np.ones(n_pos), np.zeros(10000)]
    print(f"positives {t.mean():6.1%}: ROC-AUC {roc_auc_score(t, s):.3f}   average precision {average_precision_score(t, s):.3f}   (random model's AP {t.mean():.3f})")

positives  50.0%: ROC-AUC 0.857   average precision 0.855   (random model's AP 0.500)
positives   9.1%: ROC-AUC 0.860   average precision 0.451   (random model's AP 0.091)
positives   1.0%: ROC-AUC 0.867   average precision 0.136   (random model's AP 0.010)


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [6]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, precision_score, recall_score
clf = make_pipeline(StandardScaler(), LogisticRegression())
pred = cross_val_predict(clf, df[["dist_km", "age_yrs"]], df["sold_fast"], cv=4)   # honest predictions
tn, fp, fn, tp = confusion_matrix(df["sold_fast"], pred).ravel()
print("TP", tp, "FP", fp, "FN", fn, "TN", tn)
print("precision", round(precision_score(df["sold_fast"], pred), 2), "recall", round(recall_score(df["sold_fast"], pred), 2))

TP 6 FP 1 FN 0 TN 5
precision 0.86 recall 1.0


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [7]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: Precision

Compute the precision from the confusion-matrix counts.

In [8]:
tp, fp, fn, tn = 40, 10, 20, 130
precision = None  # ← your code here
check(precision, 0.8, hint='Of everything flagged positive (TP + FP), how much was right?')

❌ Not yet. Hint: Of everything flagged positive (TP + FP), how much was right?


### Challenge 2: F1 score

Combine precision and recall into the F1 score.

In [9]:
precision, recall = 0.8, 2 / 3
f1 = None  # ← your code here
check(f1, 0.7272727272727272, hint='The harmonic mean: 2PR / (P + R).')

❌ Not yet. Hint: The harmonic mean: 2PR / (P + R).


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. Precision** — the missing line:

```python
precision = tp / (tp + fp)
```

**2. F1 score** — the missing line:

```python
f1 = 2 * precision * recall / (precision + recall)
```